# Singular Values, the SVD, and Cholesky: Class Notebook
MATH-3515 Numerical Analysis · Module 12

This notebook goes with the slides *Singular Values, the SVD, and Cholesky Factorization*. Each part starts where a slide says **→ Notebook, Part N**.

All the code is given. Run the cells in order (Runtime → Run all works too) and read the output. Each part ends with **What to notice**.

### Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
# Okabe-Ito colorblind-safe palette
OI = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]

def hilbert(n):
    i = np.arange(1, n + 1)
    return 1.0 / (i[:, None] + i[None, :] - 1)

print("Ready.")

## Part 1. The unit circle becomes an ellipse
The **image of the unit circle** under $A$: take every unit vector $\mathbf x$ ($\|\mathbf x\|_2=1$), multiply by $A$, and see where it lands. Here we map 2000 points of the circle. The arrows are $\sigma_1\mathbf u_1$ and $\sigma_2\mathbf u_2$ from the SVD: the semi-axes of the ellipse.

The plot shows the unit circle (dashed) and its image (solid), with equal axis scales, so a flat ellipse really looks flat.

In [ ]:
def circle_image(A, title, ax):
    theta = np.linspace(0, 2 * np.pi, 2000)
    circle = np.vstack([np.cos(theta), np.sin(theta)])
    image = A @ circle
    U, s, Vt = np.linalg.svd(A)
    ax.plot(circle[0], circle[1], "--", color=OI[6], label="unit circle")
    ax.plot(image[0], image[1], "-", color=OI[0], linewidth=2, label="image A x")
    for i, (col, ls) in enumerate(((OI[1], "-"), (OI[2], "--"))):
        tip = s[i] * U[:, i]
        ax.annotate("", xy=tip, xytext=(0, 0),
                    arrowprops=dict(arrowstyle="->", color=col, linewidth=2, linestyle=ls))
        ax.annotate(f"sigma_{i + 1} = {s[i]:.4g}", tip, textcoords="offset points", xytext=(5, 5))
    ax.set_aspect("equal")
    ax.set_xlabel("first coordinate")
    ax.set_ylabel("second coordinate")
    ax.grid(alpha=0.3)
    ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    return s

D = np.array([[3.0, 0.0], [0.0, 0.5]])      # diagonal: the easy case
C = np.array([[3.0, 0.0], [4.0, 5.0]])
N = np.array([[1.0, 1.0], [1.0, 1.001]])     # almost singular (Tuesday's Exercise 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for A, name, ax in ((D, "D = [[3, 0], [0, 1/2]]", axes[0]), (C, "C = [[3, 0], [4, 5]]", axes[1]),
                    (N, "N = [[1, 1], [1, 1.001]]", axes[2])):
    s = circle_image(A, name, ax)
    print(f"{name}:  sigma = {s},  ||A||_2 = {s[0]:.4f},  kappa_2 = sigma_1/sigma_2 = {s[0] / s[1]:.4g}")
plt.tight_layout()
plt.show()

**What to notice.**
- For the diagonal $D$: the ellipse is not rotated. Its semi-axes are $3$ and $\tfrac12$ along the coordinate axes, so $\sigma = 3, \tfrac12$ (the absolute values of the diagonal entries, largest first), $\|D\|_2 = 3$, $\kappa_2 = 6$. Here $U = V = I$ (up to signs) and $\Sigma = D$.
- For $C$: $\sigma_1 = 3\sqrt5 \approx 6.708$, $\sigma_2=\sqrt5\approx2.236$, $\kappa_2 = 3$. The ellipse is fat: $C$ is well-conditioned.
- For $N$: the ellipse is almost a segment. $\kappa_2 \approx 4\times10^3$, so solving $N\mathbf x=\mathbf b$ loses about 4 digits (compare Tuesday's $\kappa_\infty\approx4004$).
- $\kappa_2 = \sigma_1/\sigma_2$ is the flatness of the ellipse. For $C$ and $N$ the ellipse is tilted: the directions $\mathbf v_1, \mathbf v_2$ have to be found (Part 2).

## Part 2. How the computer finds the SVD
**(a)** `np.linalg.svd` on $C$, compared with the hand answer from the slides:
$\sigma = 3\sqrt5,\ \sqrt5$; $\ \mathbf u_1 = (1,3)/\sqrt{10}$, $\mathbf u_2 = (3,-1)/\sqrt{10}$; $\ \mathbf v_1=(1,1)/\sqrt2$, $\mathbf v_2=(1,-1)/\sqrt2$.

NumPy returns $U$, the singular values, and $V^T$ (not $V$).

In [ ]:
U, s, Vt = np.linalg.svd(C)
V = Vt.T
print("sigma from svd :", s, "   hand:", np.array([3 * np.sqrt(5), np.sqrt(5)]))
print("U from svd:\n", U)
print("U by hand:\n", np.array([[1, 3], [3, -1]]).T / np.sqrt(10))
print("V from svd:\n", V)
print("V by hand:\n", np.array([[1, 1], [1, -1]]).T / np.sqrt(2))
print("Check U Sigma V^T = C:", np.allclose(U @ np.diag(s) @ Vt, C))

**What to notice.** The singular values agree exactly. The columns of $U$ and $V$ agree **up to sign**: if $(\mathbf u, \mathbf v)$ is a pair of singular vectors, so is $(-\mathbf u, -\mathbf v)$. Both answers are correct, and both give $U\Sigma V^T = C$.

**(b)** Why the computer does *not* use $\sqrt{\text{eig}(A^TA)}$. Compare the two ways on a $3\times3$ matrix with eigenvalues 1, 2, 3 (from Moler, Ch. 10) and on Hilbert matrices.

In [ ]:
def sigma_two_ways(A):
    s_svd = np.linalg.svd(A, compute_uv=False)
    s_eig = np.sqrt(np.abs(np.sort(np.linalg.eigvalsh(A.T @ A))[::-1]))
    return s_svd, s_eig

M = np.array([[-149.0, -50, -154], [537, 180, 546], [-27, -9, -25]])
print("3 x 3 matrix: eigenvalues =", np.sort(np.linalg.eigvals(M).real))
s_svd, s_eig = sigma_two_ways(M)
print("sigma from svd            :", np.array2string(s_svd, precision=8))
print("sigma from sqrt(eig(A^T A)):", np.array2string(s_eig, precision=8))
print(f"kappa_2(A) = {s_svd[0] / s_svd[-1]:.3g},   kappa_2(A^T A) = kappa_2(A)^2 = {(s_svd[0] / s_svd[-1])**2:.3g}")
print()
print(f"{'matrix':>10s} {'kappa_2(A)':>11s} {'sigma_min by svd':>17s} {'by sqrt(eig(AtA))':>18s} {'relative difference':>20s}")
for name, A in (("3 x 3", M), ("hilbert(6)", hilbert(6)), ("hilbert(8)", hilbert(8)), ("hilbert(10)", hilbert(10))):
    s_svd, s_eig = sigma_two_ways(A)
    rel = abs(s_eig[-1] - s_svd[-1]) / s_svd[-1]
    print(f"{name:>10s} {s_svd[0] / s_svd[-1]:>11.2e} {s_svd[-1]:>17.6e} {s_eig[-1]:>18.6e} {rel:>20.1e}")

**What to notice.**
- The $3\times3$ matrix has harmless eigenvalues 1, 2, 3, yet $\kappa_2 \approx 2.8\times10^5$. **Eigenvalues do not measure sensitivity; singular values do.**
- Through $A^TA$, the smallest singular value loses about twice as many digits: $\kappa_2(A^TA)=\kappa_2(A)^2$. For `hilbert(10)` the $A^TA$ route is wrong by a factor of about $10^4$: not a single correct digit.
- `np.linalg.svd` never forms $A^TA$. It uses orthogonal factorization steps, which do not amplify rounding errors.

## Part 3. Image compression
A grayscale image is a matrix of brightness values. Keep the first $r$ rank-one pieces of its SVD:
$$A_r = \sigma_1\mathbf u_1\mathbf v_1^T + \dots + \sigma_r\mathbf u_r\mathbf v_r^T,\qquad \|A-A_r\|_2=\sigma_{r+1}.$$

The next cell loads the image from the course repository. If it cannot (for example, without internet), it uses a test image that ships with matplotlib.

In [ ]:
import io, urllib.request
import matplotlib.image as mpimg
from matplotlib.cbook import get_sample_data

URL = "https://raw.githubusercontent.com/acintron-cc/math3515-202710/main/notebooks/data/sleepyluz.jpg"
try:
    with urllib.request.urlopen(URL, timeout=10) as f:
        img = mpimg.imread(io.BytesIO(f.read()), format="jpg")
    source = "sleepyluz.jpg (course repository)"
except Exception:
    with get_sample_data("grace_hopper.jpg") as f:
        img = mpimg.imread(f, format="jpg")
    source = "grace_hopper.jpg (matplotlib sample image)"

img = np.asarray(img, dtype=float)
A_img = img[..., :3].mean(axis=2) if img.ndim == 3 else img     # grayscale: average the color channels
m, n = A_img.shape
U_img, s_img, Vt_img = np.linalg.svd(A_img, full_matrices=False)  # economy SVD: U_img is m x p, Vt_img is p x n
print(f"Image: {source},  size {m} x {n},  rank {np.linalg.matrix_rank(A_img)}")

The plot shows the singular values of the image on a logarithmic scale: a few are large, then they drop off.

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.semilogy(np.arange(1, len(s_img) + 1), s_img, "-", color=OI[0])
plt.xlabel("index k")
plt.ylabel("sigma_k (log scale)")
plt.title("Singular values of the image")
plt.grid(alpha=0.3)
plt.show()

Rank-$r$ approximations side by side with the original. Change the list `ranks` and run again.

In [ ]:
ranks = [5, 20, 50]

ranks = [r for r in ranks if 1 <= r < len(s_img)]   # keep only ranks smaller than the image's rank
print(f"ranks shown: {ranks} (each must be between 1 and {len(s_img) - 1})")

def rank_r(r):
    return (U_img[:, :r] * s_img[:r]) @ Vt_img[:r, :]

fig, axes = plt.subplots(1, len(ranks) + 1, figsize=(4 * (len(ranks) + 1), 3.6))
axes[0].imshow(A_img, cmap="gray")
axes[0].set_title(f"original (rank {len(s_img)})")
print(f"{'r':>4s} {'storage r(m+n+1)':>17s} {'% of m*n':>9s} {'error sigma_(r+1)/sigma_1':>26s}")
for ax, r in zip(axes[1:], ranks):
    ax.imshow(rank_r(r), cmap="gray")
    ax.set_title(f"rank {r}")
    storage = r * (m + n + 1)
    print(f"{r:>4d} {storage:>17d} {100 * storage / (m * n):>8.1f}% {s_img[r] / s_img[0]:>26.4f}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

**What to notice.**
- A few dozen rank-one pieces already show the picture, at a small fraction of the storage.
- The relative error in the 2-norm is exactly $\sigma_{r+1}/\sigma_1$: the singular values tell you the error before you look.
- Choosing $r$ is a trade-off between storage and error.

## Part 4. Principal components: height and weight
From Moler, *Numerical Computing with MATLAB*, Ch. 10, §10.11: "suppose we measure the height and weight of six subjects and obtain the following data." Column 1 is height, column 2 is weight (hypothetical numbers, no units).

We expect height and weight to be strongly correlated: one underlying component, call it **size**, should predict both.

In [ ]:
A = np.array([[47, 15], [93, 35], [53, 15], [45, 10], [67, 27], [42, 10]], dtype=float)
U, s, Vt = np.linalg.svd(A, full_matrices=False)
V = Vt.T
if V[0, 0] < 0:                 # singular vectors are defined up to sign; match Moler's signs
    U[:, 0] *= -1
    V[:, 0] *= -1

E1 = s[0] * np.outer(U[:, 0], V[:, 0])
size = s[0] * U[:, 0]           # one number per subject
print("sigma =", s)
print("V =\n", V)
print("E1 = sigma_1 u_1 v_1^T =\n", E1)
print("||A - E1||_2 =", np.linalg.norm(A - E1, 2), " = sigma_2")
print("size = sigma_1 u_1 =", size)
print(f"height ~ size * {V[0, 0]:.4f},   weight ~ size * {V[1, 0]:.4f}")

The plot repeats Moler's Figure 10.8: for each subject, the measured value (dark, solid bars) next to the one-component approximation (light, hatched bars), for height and for weight.

In [ ]:
idx = np.arange(1, 7)
fig, axes = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
for k, (ax, name) in enumerate(zip(axes, ("height", "weight"))):
    ax.bar(idx - 0.2, A[:, k], width=0.4, color=OI[0], label="data")
    ax.bar(idx + 0.2, E1[:, k], width=0.4, color=OI[5], hatch="//", edgecolor=OI[6], label="one component (size)")
    ax.set_ylabel(name)
    ax.set_title(name)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis="y")
axes[1].set_xlabel("subject")
plt.tight_layout()
plt.show()

**What to notice.**
- $\sigma_1 = 156.44$ is much larger than $\sigma_2 = 8.77$, so the rank-one piece $E_1$ is already close to $A$: $\|A-E_1\|_2 = \sigma_2$.
- Each subject is summarized by **one number**, its size, instead of two measurements: height $\approx 0.9468\cdot$size, weight $\approx 0.3219\cdot$size. That is **dimensionality reduction**, the idea behind principal component analysis (PCA).
- In practice the data are often standardized first (subtract each column's mean, divide by its standard deviation), as Moler notes; this example uses the data as measured.

## Part 5. Cholesky factorization
`np.linalg.cholesky(A)` returns the lower triangular $L$ with $A = LL^T$. It works only for symmetric positive definite (SPD) matrices.

In [ ]:
A = np.array([[4.0, 2.0], [2.0, 5.0]])
L = np.linalg.cholesky(A)
print("L =\n", L, "\nL L^T == A:", np.allclose(L @ L.T, A))

B = np.array([[1.0, 2.0], [2.0, 3.0]])          # symmetric, positive diagonal, NOT SPD
print("\neigenvalues of B:", np.linalg.eigvalsh(B))
try:
    np.linalg.cholesky(B)
except np.linalg.LinAlgError as err:
    print("cholesky(B) fails:", err)

H3 = hilbert(3)
print("\nH3 is SPD: eigenvalues", np.linalg.eigvalsh(H3))
print("cholesky(H3) =\n", np.linalg.cholesky(H3))
print(f"but kappa_inf(H3) = {np.linalg.cond(H3, np.inf):.0f}")

**What to notice.**
- $L = \begin{pmatrix}2&0\\1&2\end{pmatrix}$, as on the slides.
- For $B$ the factorization breaks down ($\sqrt{3-4}$): Cholesky is also a **test** for SPD.
- $H_3$ is SPD, yet $\kappa_\infty(H_3) = 748$: **SPD does not mean well-conditioned.**
- More practice: lab *Measuring Size and Sensitivity*, Topic 10, and Problem Set 2, Exercise 5.